# 00 - RDKit'in Projedeki Rolü

Bu notebook, RDKit'i bu projenin bağlamında tanıtır: **molekül yapısını Python'da temsil etme** ve kimyasal veriyi ileride makine öğrenmesine hazırlanabilir hâle getirme aracı.

Bu çalışma bir bilimsel pipeline değildir. Bu notebookta gerçek veri indirme, final dataset seçimi, modelleme, train/test split, metric/evaluation, Macro F1, benchmark veya bilimsel sonuç üretimi yapılmaz.


## Öğrenme hedefleri

Bu notebookun sonunda şunları ayırt edebilmelisin:

- RDKit'in ¹³C NMR → fonksiyonel grup tahmini projesindeki yardımcı rolü.
- SMILES metninin RDKit `Mol` nesnesine nasıl dönüştüğünü.
- Descriptor, fingerprint ve SMARTS kavramlarının ne işe yaradığını.
- Label matrix fikrinin, bir molekül için birden fazla etiket taşıyabilme mantığıyla nasıl ilişkili olduğunu.
- Oyuncak örnek ile gerçek proje datası arasındaki farkı.


## RDKit bu projede ne yapar?

Projenin bilimsel hedefi ¹³C NMR spektrumlarından fonksiyonel grup tahmini yapmaktır. RDKit burada doğrudan "spektrumu anlayan" bir model değildir. RDKit daha çok molekül yapısı tarafında yardımcı olur.

RDKit ile yapılabilecek hazırlık işleri şunlara benzer:

- SMILES veya benzeri yapı gösterimlerini Python içinde molekül nesnesine çevirmek.
- Molekülün atom, bağ, halka, aromatiklik gibi yapısal özelliklerini incelemek.
- Basit descriptor'lar ve fingerprint'ler üretmenin ne anlama geldiğini öğrenmek.
- SMARTS desenleri ile belirli alt yapıları aramayı kavramak.
- Fonksiyonel grup etiketlerinin ileride nasıl bir label matrix'e dönüşebileceğini düşünmek.

Bu notebook yalnızca bu kavramları tanıtır. Final label schema, veri kaynağı kararı veya modelleme kararı vermez.


In [ ]:
# RDKit bu ortamda kurulu olmayabilir.
# Bu yüzden import işlemini güvenli yapıyoruz: RDKit yoksa notebook tamamen çökmesin.

RDKit_AVAILABLE = False
rdkit_import_error = None

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    try:
        from rdkit.Chem import rdFingerprintGenerator
    except Exception:
        rdFingerprintGenerator = None
    RDKit_AVAILABLE = True
except Exception as exc:
    rdkit_import_error = exc

if RDKit_AVAILABLE:
    print("RDKit bulundu. Kod hücreleri çalıştırılabilir.")
else:
    print("RDKit bu ortamda bulunamadı.")
    print("Notebook kavramsal olarak okunabilir; RDKit gerektiren hücreler güvenli şekilde atlanacak.")
    print(f"Import hatası: {type(rdkit_import_error).__name__}: {rdkit_import_error}")


## Küçük oyuncak molekül listesi

İlk notebooklarda gerçek NMReDATA dosyasını ana veri olarak kullanmıyoruz. Bunun yerine küçük, kontrol edilebilir ve kolay yorumlanabilir moleküllerle başlıyoruz.

Bu liste final proje dataset'i değildir. Sadece RDKit kavramlarını görmek için seçilmiş oyuncak örneklerdir.


In [ ]:
toy_molecules = [
    {"name": "Etanol", "smiles": "CCO", "note": "Basit alkol örneği"},
    {"name": "Aseton", "smiles": "CC(=O)C", "note": "Basit karbonil örneği"},
    {"name": "Asetik asit", "smiles": "CC(=O)O", "note": "Karboksilik asit örneği"},
    {"name": "Benzen", "smiles": "c1ccccc1", "note": "Aromatik halka örneği"},
    {"name": "Etilamin", "smiles": "CCN", "note": "Basit amin örneği"},
]

def print_rows(rows, columns):
    """Küçük öğretici tabloları ek paket kullanmadan yazdırır."""
    widths = []
    for column in columns:
        values = [str(row.get(column, "")) for row in rows]
        widths.append(max([len(column)] + [len(value) for value in values]))

    header = " | ".join(column.ljust(width) for column, width in zip(columns, widths))
    line = "-+-".join("-" * width for width in widths)
    print(header)
    print(line)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(width) for column, width in zip(columns, widths)))

print_rows(toy_molecules, ["name", "smiles", "note"])


## SMILES → Mol object

SMILES, molekül yapısını tek satırlık metin olarak yazmanın yaygın bir yoludur. RDKit bu metni okuyup bir `Mol` nesnesine çevirebilir.

`Mol` nesnesi, molekülü Python içinde atomlar, bağlar ve yapısal ilişkilerle temsil eder. Böylece metin olarak duran kimyasal bilgi, programla incelenebilir hâle gelir.


In [ ]:
parsed_molecules = []

if RDKit_AVAILABLE:
    for item in toy_molecules:
        mol = Chem.MolFromSmiles(item["smiles"])
        parsed_molecules.append({**item, "mol": mol})

    rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        rows.append({
            "name": item["name"],
            "input_smiles": item["smiles"],
            "canonical_smiles": Chem.MolToSmiles(mol) if mol is not None else "OKUNAMADI",
            "atom_count": mol.GetNumAtoms() if mol is not None else "-",
            "bond_count": mol.GetNumBonds() if mol is not None else "-",
        })

    print_rows(rows, ["name", "input_smiles", "canonical_smiles", "atom_count", "bond_count"])
else:
    print("RDKit olmadığı için SMILES -> Mol dönüşümü bu ortamda çalıştırılmadı.")
    print("Kavramsal fikir: SMILES metni, RDKit içinde programla incelenebilir bir Mol nesnesine dönüşür.")


## Descriptor nedir?

Descriptor, molekülün sayısal veya kategorik bir özetidir. Örneğin molekül ağırlığı, atom sayısı, halka sayısı veya hidrojen bağı verici/alıcı sayısı birer descriptor olabilir.

Bu notebookta descriptor'lar yalnızca kavram olarak gösterilir. Bu değerler model girdisi olarak kullanılmaz, karşılaştırmalı sonuç veya benchmark üretilmez.


In [ ]:
if RDKit_AVAILABLE:
    descriptor_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        descriptor_rows.append({
            "name": item["name"],
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "mol_weight": round(Descriptors.MolWt(mol), 2),
            "h_donor": rdMolDescriptors.CalcNumHBD(mol),
            "h_acceptor": rdMolDescriptors.CalcNumHBA(mol),
        })

    print_rows(descriptor_rows, ["name", "formula", "mol_weight", "h_donor", "h_acceptor"])
else:
    print("RDKit olmadığı için descriptor örneği çalıştırılmadı.")


## Fingerprint nedir?

Fingerprint, molekül yapısını sabit uzunlukta bit dizisi gibi temsil etmeye yarayan yapısal bir özettir. Bir bitin açık olması, belirli bir yerel yapı parçasının molekülde bulunduğunu gösterebilir.

Fingerprint kavramı ileride makine öğrenmesiyle ilişkilendirilebilir; fakat bu notebookta model kurulmaz ve performans skoru üretilmez. Burada yalnızca "molekül yapısı sayısal temsile nasıl yaklaşır?" sorusuna giriş yapıyoruz.


In [ ]:
if RDKit_AVAILABLE and rdFingerprintGenerator is not None:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=128)
    fingerprint_rows = []

    for item in parsed_molecules:
        mol = item["mol"]
        fingerprint = generator.GetFingerprint(mol)
        on_bits = list(fingerprint.GetOnBits())
        fingerprint_rows.append({
            "name": item["name"],
            "on_bit_count": len(on_bits),
            "first_on_bits": on_bits[:8],
        })

    print_rows(fingerprint_rows, ["name", "on_bit_count", "first_on_bits"])
elif RDKit_AVAILABLE:
    print("Bu RDKit kurulumunda rdFingerprintGenerator bulunamadı; fingerprint örneği atlandı.")
else:
    print("RDKit olmadığı için fingerprint örneği çalıştırılmadı.")


## SMARTS nedir?

SMARTS, molekül içinde belirli alt yapıları aramak için kullanılan desen dilidir. SMILES bir molekülü yazmaya yararken, SMARTS bir yapı motifini aramaya yarar.

Örneğin alkol, karbonil, karboksilik asit veya aromatik halka gibi basit desenler SMARTS ile aranabilir. Aşağıdaki desenler öğretici oyuncak örneklerdir; final fonksiyonel grup label schema değildir.


In [ ]:
toy_smarts_patterns = [
    {"label": "oyuncak_alkol", "smarts": "[OX2H]"},
    {"label": "oyuncak_karbonil", "smarts": "[CX3]=[OX1]"},
    {"label": "oyuncak_karboksilik_asit", "smarts": "C(=O)[OX2H1]"},
    {"label": "oyuncak_aromatik_halka", "smarts": "c1ccccc1"},
    {"label": "oyuncak_amin", "smarts": "[NX3]"},
]

if RDKit_AVAILABLE:
    pattern_rows = []
    for pattern in toy_smarts_patterns:
        smarts_mol = Chem.MolFromSmarts(pattern["smarts"])
        pattern_rows.append({
            "label": pattern["label"],
            "smarts": pattern["smarts"],
            "valid": smarts_mol is not None,
        })

    print_rows(pattern_rows, ["label", "smarts", "valid"])
else:
    print("RDKit olmadığı için SMARTS desenleri doğrulanmadı.")
    print_rows(toy_smarts_patterns, ["label", "smarts"])


## Label matrix fikri

Fonksiyonel grup tahmini çok etiketli (multi-label) bir problem olarak düşünülür: bir molekülde aynı anda birden fazla fonksiyonel grup bulunabilir.

Bu fikir tablo hâlinde şöyle temsil edilebilir: satırlar moleküller, sütunlar etiketlerdir. Hücredeki `1`, ilgili etiketin oyuncak kurala göre bulunduğunu; `0`, bulunmadığını gösterir.

Aşağıdaki tablo sadece öğretici bir oyuncak label matrix'tir. Final proje label schema değildir, model girdisi değildir ve bilimsel sonuç değildir.


In [ ]:
if RDKit_AVAILABLE:
    compiled_patterns = []
    for pattern in toy_smarts_patterns:
        compiled_patterns.append({
            "label": pattern["label"],
            "query": Chem.MolFromSmarts(pattern["smarts"]),
        })

    matrix_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        row = {"name": item["name"]}
        for pattern in compiled_patterns:
            query = pattern["query"]
            row[pattern["label"]] = int(query is not None and mol.HasSubstructMatch(query))
        matrix_rows.append(row)

    matrix_columns = ["name"] + [pattern["label"] for pattern in toy_smarts_patterns]
    print_rows(matrix_rows, matrix_columns)
else:
    print("RDKit olmadığı için oyuncak label matrix otomatik üretilmedi.")
    print("Kavramsal fikir: satırlar moleküller, sütunlar fonksiyonel grup etiketleri olur.")


## ¹³C NMR projesiyle bağlantı

Bu projenin ana yönü ¹³C NMR spektrumlarından fonksiyonel grup tahminidir. RDKit'in burada öğrettiği taraf, molekül yapısını güvenli ve denetlenebilir biçimde ele almaktır.

İleride RDKit şu tür düşünme adımlarında yardımcı olabilir:

- Molekül yapısından fonksiyonel grup var/yok etiketlerinin nasıl türetilebileceğini anlamak.
- Yakın veya duplicate molekül risklerini tartışmak.
- Kimyasal yapının metadata ve spektrum bilgisiyle nasıl ilişkilendirilebileceğini görmek.
- Notebooklarda hata toleranslı okuma ve kalite kontrol alışkanlığı kazanmak.

Bu notebookta `nmrshiftdb2rawdata.nmredata.sd` dosyası ana veri olarak kullanılmaz. O dosya yalnızca ileride, daha gerçekçi SDF/NMReDATA okuma ve debugging öğrenimi için `learning sample / pilot example` statüsünde kullanılacaktır.


## Bu notebookta bilinçli olarak yapılmayanlar

- Gerçek veri indirme yapılmadı.
- Final dataset seçimi yapılmadı.
- `data/raw/` veya `data/processed/` kullanılmadı.
- Model kurulmadı.
- Train/test split yapılmadı.
- Macro F1, accuracy, benchmark veya skor üretilmedi.
- ADR kabulü veya learning gate geçişi yapılmadı.

Bu sınırlar, Command Center'ın güvenli öğrenme ve hazırlık rolünü korumak içindir.


In [ ]:
notebook_boundary_check = {
    "uses_toy_molecules_only": True,
    "downloads_data": False,
    "trains_model": False,
    "creates_train_test_split": False,
    "computes_metrics_or_benchmark": False,
    "selects_final_dataset": False,
}

for key, value in notebook_boundary_check.items():
    print(f"{key}: {value}")


## Kapanış

RDKit'i bu aşamada bir modelleme aracı gibi değil, molekül yapısını anlamayı ve denetlenebilir kimyasal veri hazırlığı düşüncesini destekleyen bir araç gibi okumak en sağlıklı başlangıçtır.

Bir sonraki adımda amaç, basit molekül temsillerini daha rahat okumak ve RDKit nesnelerinin hangi bilgileri taşıdığını daha yakından incelemek olabilir.
